In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.6 Batching and padding

Training wants many sequences at once. Contiguous windows from one long stream
need no padding; ragged inputs (different lengths) must be padded, with a mask
so the model ignores the fake pad tokens.

In [ ]:
from pocketlm import CharTokenizer, make_batch, pad_batch

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
x, y = make_batch(data, block_size=16, batch_size=4,
                  generator=torch.Generator().manual_seed(0))
print("x", tuple(x.shape), " y", tuple(y.shape), "(y is x shifted by one)")

padded, mask = pad_batch([tok.encode("hi"), tok.encode("hello")], pad_id=0)
print("padded:\n", padded)
print("mask (True = real token):\n", mask)

**Mask leakage** is the classic bug: forget the mask and the model trains on,
and attends to, padding, learning from tokens that were never there. Always
carry the mask alongside the padded batch.

In [ ]:
assert tuple(x.shape) == (4, 16)
assert padded.shape == mask.shape
assert int(mask[0].sum()) == len(tok.encode("hi"))